# Create Maudsley Charity Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

Maudsley Charity is a UK mental-health charity supporting services, community programmes, and research connected to South London and Maudsley NHS Foundation Trust, King's College London, and related partners. This ingest covers the charity's published 360Giving grant record, one row per unique grant.

**Method 1 (direct open-data download).** The 360Giving Data Registry lists publisher `Maudsley Charity` and resolves to a direct Excel workbook on `maudsleycharity.org` (`MC-360-Giving-26-11-2025v2.xlsx`). No browser automation or search API is required.

**Awarding body:** Maudsley Charity - F4320313345 (GB, ROR 028vg3q27, DOI 10.13039/100012176)

**Source:** the workbook's `grants` sheet. This is an **org-level grant funder**: each grant is made to a recipient organization, with no named principal investigator in the source. The source provides an `Identifier`, title, description, GBP amount, real `Award Date`, planned grant start/end dates, recipient organization/country, and grant programme.

**Schema choices:**
- One row per unique grant. The source workbook has 367 rows but 12 exact duplicate rows by `Identifier`; the script skips exact repeats and raises on conflicting duplicate IDs, producing 355 awards.
- `funder_award_id` = `360G-maudsley-` plus the source `Identifier`, because Maudsley's 360Giving identifiers are bare numeric values in the workbook.
- `display_name` = source `Title`; `description` = source `Description`.
- `funding_type` = `grant` uniformly; `funder_scheme` = `Grant Programme:Title`.
- `amount` = positive `Amount Awarded`; `currency` = source `Currency` where amount is positive. Blank/0 means NULL, never imputed. Amount coverage is 99.4%, so runbook section 6.7 is not waived.
- `start_date` = planned start date where present, otherwise award date; `end_date` = planned end date where present. These are source-published day-level dates, not fabricated year-only dates.
- `lead_investigator` = org-only lead: given/family/orcid NULL and `affiliation.name` = recipient organization. `affiliation.country` is mapped only from the explicit source `Recipient Org:Country` value.
- `landing_page_url` is NULL because the workbook does not expose per-grant pages.

**Contractor handoff:** this notebook is prepared for admin execution after the parquet is uploaded to S3. Contractor has no S3/Databricks access.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.maudsley_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/maudsley/maudsley_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.maudsley_raw;

## Step 1.5: Inspect raw coverage, dates, programmes, and amounts

Expected after local validation: 355 unique grants, start_year 2013-2026, end_year 2019-2030, title/description/recipient/programme/start_year 100%, amount/currency 99.4%.

In [ ]:
%sql
DESCRIBE openalex.awards.maudsley_raw;

In [ ]:
%sql
SELECT *
FROM openalex.awards.maudsley_raw
LIMIT 5;

In [ ]:
%sql
-- Uniqueness and high-level coverage from raw staging.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(title) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(recipient_org) AS has_recipient_org,
    COUNT(recipient_country_iso) AS has_recipient_country,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.maudsley_raw;

In [ ]:
%sql
-- Programme/year distribution and amount coverage.
SELECT
    grant_programme,
    start_year,
    COUNT(*) AS rows,
    COUNT(amount) AS has_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.maudsley_raw
GROUP BY grant_programme, start_year
ORDER BY start_year DESC, rows DESC;

In [ ]:
%sql
-- Top recipient organizations, to confirm org-level lead parsing.
SELECT recipient_org, recipient_country_iso, COUNT(*) AS rows,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.maudsley_raw
WHERE recipient_org IS NOT NULL
GROUP BY recipient_org, recipient_country_iso
ORDER BY rows DESC, total_gbp DESC
LIMIT 20;

## Step 1.6: Fail-fast - verify Maudsley Charity funder row exists

Runbook section 2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320313345;

## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.maudsley_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320313345  -- Maudsley Charity
), awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        COALESCE(n.title, CONCAT('Maudsley Charity grant ', n.source_identifier)) AS display_name,
        n.description,
        f.funder_id,
        n.funder_award_id,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN n.currency ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'grant' AS funding_type,
        n.grant_programme AS funder_scheme,
        'maudsley_360giving' AS provenance,
        TRY_CAST(n.start_date AS DATE) AS start_date,
        TRY_CAST(n.end_date AS DATE) AS end_date,
        TRY_CAST(n.start_year AS INT) AS start_year,
        TRY_CAST(n.end_year AS INT) AS end_year,
        CASE
            WHEN n.recipient_org IS NOT NULL THEN struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    n.recipient_org AS name,
                    n.recipient_country_iso AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
            ELSE NULL
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.source_url AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               TRY_CAST(abs(xxhash64(CONCAT(
                   TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
               ))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM openalex.awards.maudsley_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    * EXCEPT(start_year, end_year),
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE start_year END AS start_year,
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE end_year END AS end_year
FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw at priority 184

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'maudsley_360giving' AND priority = 184;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    184 AS priority  -- Maudsley Charity grants
FROM openalex.awards.maudsley_awards;

## Step 6: Verification

Amount coverage should be about 99.4% and currency should be GBP. Country is source-authoritative from `Recipient Org:Country`, not inferred. Planned end years can legitimately run beyond current year + 1 when paired with an in-range start year.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.maudsley_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.maudsley_awards;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(end_date), COUNT(*)) * 100.0, 1) AS pct_end_date
FROM openalex.awards.maudsley_awards;

In [ ]:
%sql
-- Runbook section 6.7 amount/currency check. Expected: 353/355 rows, GBP only.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.maudsley_awards;

In [ ]:
%sql
-- Duplicate guard.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.maudsley_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Year sanity. max_start_year should be <= YEAR(current_date()) + 1 after inline cap.
SELECT
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    MIN(end_year) AS min_end_year,
    MAX(end_year) AS max_end_year,
    SUM(CASE WHEN start_year > YEAR(current_date()) + 1 THEN 1 ELSE 0 END) AS future_start_rows
FROM openalex.awards.maudsley_awards;

In [ ]:
%sql
-- Programme split.
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    COUNT(amount) AS has_amount,
    ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.maudsley_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;

In [ ]:
%sql
-- Country sanity: lead_investigator.affiliation.country is source-authoritative.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.maudsley_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.maudsley_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample rows for admin QA.
SELECT
    id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_scheme,
    funding_type,
    start_date,
    end_date,
    amount,
    currency,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS country
FROM openalex.awards.maudsley_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;

## Handoff notes

Contractor-complete status: script and notebook are ready, `CreateAwards.ipynb` has priority 184, and the tracker is marked Step 4 in the PR. Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect the verification cells, and only then decide whether to wire job YAML.
